In [1]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tkinter
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier

candidates = [
    Path("../data/HR_Attrition_Indian_Dataset.csv"),
    Path("data/HR_Attrition_Indian_Dataset.csv"),
    Path("../HR_Attrition_Indian_Dataset.csv"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None: raise FileNotFoundError("Put dataset in project data/ folder.")
df = pd.read_csv(DATA_PATH).drop_duplicates()
df = df[df["LeftCompany"].isin(["Yes","No"])].copy()
if "DateOfJoining" in df:
    df["JoiningYear"] = pd.to_datetime(df["DateOfJoining"], errors="coerce").dt.year
    df = df.drop(columns=["DateOfJoining"])
X = df.drop(columns=[c for c in ["EmployeeID","FullName","LeftCompany"] if c in df])
y = df["LeftCompany"].map({"No":0,"Yes":1})
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
num = X.select_dtypes(include=np.number).columns.tolist()
cat = X.select_dtypes(exclude=np.number).columns.tolist()
prep_sparse = ColumnTransformer([
 ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),
 ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),cat)
])
prep_dense = ColumnTransformer([
 ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),
 ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),cat)
])

In [2]:
estimators = {
 "Logistic Regression": (LogisticRegression(max_iter=2000,class_weight="balanced",random_state=42), prep_sparse),
 "Decision Tree": (DecisionTreeClassifier(max_depth=8,class_weight="balanced",random_state=42), prep_sparse),
 "Random Forest": (RandomForestClassifier(n_estimators=250,class_weight="balanced",random_state=42,n_jobs=-1), prep_sparse),
 "Neural Network (MLP)": (MLPClassifier(hidden_layer_sizes=(64,32),max_iter=500,early_stopping=True,random_state=42), prep_sparse),
 "Gaussian Naive Bayes": (GaussianNB(), prep_dense),
 "Linear Discriminant Analysis": (LinearDiscriminantAnalysis(solver="lsqr",shrinkage="auto"), prep_sparse),
}
try:
 from xgboost import XGBClassifier
 estimators["XGBoost"]=(XGBClassifier(n_estimators=250,max_depth=4,learning_rate=.05,subsample=.9,colsample_bytree=.9,eval_metric="logloss",random_state=42,n_jobs=-1),prep_sparse)
except Exception as e:
 print("XGBoost unavailable:",e)
rows=[]; pipelines={}
for name,(est,prep) in estimators.items():
 pipe=Pipeline([("preprocess",prep),("model",est)])
 try:
  pipe.fit(X_train,y_train); p=pipe.predict(X_test); proba=pipe.predict_proba(X_test)[:,1]
  rows.append({"Model":name,"Accuracy":accuracy_score(y_test,p),"Precision":precision_score(y_test,p,zero_division=0),"Recall":recall_score(y_test,p,zero_division=0),"F1":f1_score(y_test,p,zero_division=0),"ROC-AUC":roc_auc_score(y_test,proba)})
  pipelines[name]=pipe
 except Exception as e: print("Skipped",name,e)
results=pd.DataFrame(rows).sort_values("ROC-AUC",ascending=False)
display(results.round(4))

XGBoost unavailable: No module named 'xgboost'


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
2,Random Forest,0.781,0.7923,0.5553,0.6529,0.7513
3,Neural Network (MLP),0.718,0.6846,0.4447,0.5392,0.7142
1,Decision Tree,0.741,0.6739,0.5849,0.6263,0.6947
5,Linear Discriminant Analysis,0.699,0.6378,0.4367,0.5184,0.6937
0,Logistic Regression,0.643,0.5148,0.6577,0.5775,0.6932
4,Gaussian Naive Bayes,0.671,0.5772,0.4232,0.4883,0.6711


In [3]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scores_by_model=[]
for name,(est,prep) in estimators.items():
 pipe=Pipeline([("preprocess",prep),("model",est)])
 try:
  scores=cross_validate(pipe,X_train,y_train,cv=cv,scoring={"accuracy":"accuracy","precision":"precision","recall":"recall","f1":"f1","roc_auc":"roc_auc"},n_jobs=1)
  scores_by_model.append({"Model":name,**{f"{k}_mean":scores["test_"+k].mean() for k in ["accuracy","precision","recall","f1","roc_auc"]}})
 except Exception as e: print("CV skipped",name,e)
cv_results=pd.DataFrame(scores_by_model).sort_values("roc_auc_mean",ascending=False)
display(cv_results.round(4))

,Model,accuracy_mean,precision_mean,recall_mean,f1_mean,roc_auc_mean
2,Random Forest,0.8010,0.8276,0.5856,0.6853,0.7702
3,Neural Network (MLP),0.7255,0.6741,0.5081,0.5775,0.7326
1,Decision Tree,0.7397,0.6645,0.6085,0.6345,0.7206
0,Logistic Regression,0.6468,0.5191,0.6475,0.5759,0.7010
5,Linear Discriminant Analysis,0.6960,0.6368,0.4218,0.5069,0.7009
4,Gaussian Naive Bayes,0.6640,0.5665,0.4110,0.4758,0.6752
